In [1]:
!pip install datasets spacy pycountry pandas tqdm
!python -m spacy download en_core_web_trf  

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 44.3 MB/s  0:00:09:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


In [2]:
import spacy
import pycountry
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from collections import Counter

MODEL = "en_core_web_lg"
MAX_ARTICLES = 5000
BATCH_SIZE = 64

# Codes to always discard — known false positives
BLOCKLIST = {"VI", "GE", "LR", "MC", "GS", "UM", "TF", "IO", "SH"}

# Minimum character length for a mention to be trusted
MIN_MENTION_LENGTH = 4

CUSTOM_MAP = {
    # Abbreviations
    "US": "United States", "USA": "United States", "America": "United States",
    "UK": "United Kingdom", "Britain": "United Kingdom", "England": "United Kingdom",
    "UAE": "United Arab Emirates",
    # Official name mismatches
    "Russia": "Russian Federation", "Iran": "Iran, Islamic Republic of",
    "Syria": "Syrian Arab Republic", "North Korea": "Korea, Democratic People's Republic of",
    "South Korea": "Korea, Republic of", "Vietnam": "Viet Nam",
    "Bolivia": "Bolivia, Plurinational State of", "Venezuela": "Venezuela, Bolivarian Republic of",
    # Demonyms
    "American": "United States", "British": "United Kingdom", "French": "France",
    "German": "Germany", "Chinese": "China", "Russian": "Russian Federation",
    "Iraqi": "Iraq", "Afghan": "Afghanistan", "Iranian": "Iran, Islamic Republic of",
    "Israeli": "Israel", "Pakistani": "Pakistan", "Indian": "India",
    "Australian": "Australia", "Canadian": "Canada", "Mexican": "Mexico",
    "Colombian": "Colombia", "Brazilian": "Brazil", "Saudi": "Saudi Arabia",
    # Explicit suppressions
    "Georgia": None, "Virgin": None, "Virgin Islands": None,
    "Liberia": None,   # too many false positives — add back if you need it
    "Civil": None, "Monaco": None,
}

def resolve_country(mention):
    mention = mention.strip()

    # drop short mentions — single words under 4 chars are too ambiguous
    if len(mention) < MIN_MENTION_LENGTH:
        return None

    # check custom map
    if mention in CUSTOM_MAP:
        mapped = CUSTOM_MAP[mention]
        if mapped is None:
            return None
        mention = mapped

    # exact lookup
    try:
        c = pycountry.countries.lookup(mention)
        if c.alpha_2 in BLOCKLIST:
            return None
        return c.alpha_2
    except LookupError:
        pass

    # fuzzy lookup — only trust it if the mention is long enough to be meaningful
    if len(mention) >= 6:
        try:
            c = pycountry.countries.search_fuzzy(mention)[0]
            if c.alpha_2 in BLOCKLIST:
                return None
            return c.alpha_2
        except LookupError:
            pass

    return None


nlp = spacy.load(MODEL)

ds = load_dataset("cnn_dailymail", "3.0.0", split="train")
ds = ds.select(range(MAX_ARTICLES))

texts = [row["article"] for row in ds]
ids   = [row["id"]      for row in ds]

results = []
for i, doc in enumerate(tqdm(
    nlp.pipe(texts, batch_size=BATCH_SIZE, disable=["parser", "senter"]),
    total=len(texts)
)):
    counts = Counter()
    for ent in doc.ents:
        if ent.label_ in ("GPE", "LOC"):
            code = resolve_country(ent.text)
            if code:
                counts[code] += 1

    primary = counts.most_common(1)[0][0] if counts else None
    results.append({
        "id": ids[i],
        "primary_country": primary,
        "country_freq": dict(counts)
    })

df = pd.DataFrame(results)
df.to_parquet("ner_results.parquet", index=False)
print(df["primary_country"].value_counts().head(20))

/Users/user/GroupProjectV/Voices-in-the-news/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 5000/5000 [08:48<00:00,  9.45it/s]


primary_country
US    1959
GB     485
IQ     208
PK     119
CN     106
IN     106
MX     101
SO      84
AF      71
IR      67
ES      65
IL      60
DE      54
FR      54
IT      50
ZW      49
RU      47
ZA      40
JP      39
CA      36
Name: count, dtype: int64


,id,primary_country,country_freq
0,42c027e4ff9730fbb3de84c1af0d2c506e41c3e4,GB,"{'GB': 3, 'CO': 1}"
1,ee8871b15c50d0db17b0179a6d2beab35065f1e9,US,{'US': 1}
2,06352019a19ae31e527f37f7571c6dd7f0c5da37,US,{'US': 2}
3,24521a2abb2e1f5e34e6824e0f9e56904a2b0e88,LR,"{'US': 2, 'LR': 3, 'ZW': 1}"
4,7fe70cc8b12fab2d0a258fababf7d9c6b5e1262a,US,"{'US': 8, 'GB': 2, 'GE': 1}"
5,a1ebb8bb4d370a1fdf28769206d572be60642d70,IQ,"{'IQ': 5, 'US': 2, 'VI': 1}"
6,7c0e61ac829a3b3b653e2e3e7536cc4881d1f264,IQ,{'IQ': 7}
7,f0d73bdab711763e745cdc75850861c9018f235d,VI,"{'CO': 5, 'VI': 7, 'VE': 1}"
8,5e22bbfc7232418b8d2dd646b952e404df5bd048,US,{'US': 1}
9,613d6311ec2c1985bd44707d1796d275452fe156,US,{'US': 2}


In [4]:
import random

# pick 20 random indices from outside the first 5000 we already processed
random_indices = random.sample(range(5000, 50000), 20)

ds_sample = load_dataset("cnn_dailymail", "3.0.0", split="train")
ds_sample = ds_sample.select(random_indices)

texts = [row["article"] for row in ds_sample]
ids   = [row["id"]      for row in ds_sample]

results_sample = []
for i, doc in enumerate(tqdm(
    nlp.pipe(texts, batch_size=20, disable=["parser", "senter"]),
    total=len(texts)
)):
    counts = Counter()
    for ent in doc.ents:
        if ent.label_ in ("GPE", "LOC"):
            code = resolve_country(ent.text)
            if code:
                counts[code] += 1

    primary = counts.most_common(1)[0][0] if counts else None
    results_sample.append({
        "id": ids[i],
        "primary_country": primary,
        "country_freq": dict(counts),
        "article_text": texts[i]
    })

df_sample = pd.DataFrame(results_sample)

for _, row in df_sample.iterrows():
    print("=" * 80)
    print(f"Primary country: {row['primary_country']}")
    print(f"Country freq:    {row['country_freq']}")
    print(f"Article snippet: {row['article_text'][:800]}")
    print()

100%|██████████| 20/20 [00:02<00:00,  6.83it/s]

Primary country: SY
Country freq:    {'JO': 2, 'SY': 8}
Article snippet: (CNN) -- Feet stumbling in the pitch darkness over the uneven ground we make our way with a group of women to one of the bathrooms in the Zaatari camp. "There is no light, if we come in here there could be a guy hiding or something." one woman says. None of them want to be identified. They carry fear of the regime with them, even as they seek refuge across the border in Jordan. But "safety" is a relative term. For Syria's female refugee population, it has meant trading fear of death in their homeland for fear of something many consider to be worse: rape. There have been various stories of sexual harassment and rape in Zaatari camp -- teeming with masses who continue to stream across the border. This dark underbelly of crisis has led to a disturbing growing phenomenon: "sutra" marriages, or

Primary country: nan
Country freq:    {}
Article snippet: (CNN) -- Evan Lysacek became a household name in February when he w